In [1]:
# !wget https://cdn.netbiol.org/tflink/download_files/TFLink_Mus_musculus_interactions_All_simpleFormat_v1.0.tsv.gz

In [2]:
# !gunzip TFLink_Mus_musculus_interactions_All_simpleFormat_v1.0.tsv.gz

In [3]:
import os
import pandas as pd
import scanpy as sc

In [4]:
# ---------- Step 1: Define directory and interaction categories ----------
tf_dir = "./tf_link_data"
os.makedirs(tf_dir, exist_ok=True)

In [5]:
# First, let's check what files we have and read the actual TFLink data
print("Files in current directory:")
for file in os.listdir('.'):
    if file.endswith('.tsv'):
        print(f"Found TSV file: {file}")

# Read the TFLink data file
tflink_file = "TFLink_Mus_musculus_interactions_All_simpleFormat_v1.0.tsv"

if os.path.exists(tflink_file):
    print(f"\nReading {tflink_file}...")
    
    # Read the TSV file - let's first check the structure
    print("First few lines of the file:")
    with open(tflink_file, 'r') as f:
        for i, line in enumerate(f):
            if i < 5:  # Show first 5 lines
                print(f"Line {i+1}: {line.strip()}")
            else:
                break
    
    # Now read the full file with pandas
    print(f"\nReading full dataset...")
    df_full = pd.read_csv(tflink_file, sep='\t', low_memory=False)
    
    print(f"Full dataset shape: {df_full.shape}")
    print(f"Columns: {list(df_full.columns)}")
    print(f"\nFirst few rows:")
    print(df_full.head())
    
else:
    print(f"File {tflink_file} not found. Please download it first.")

Files in current directory:
Found TSV file: TFLink_Mus_musculus_interactions_All_simpleFormat_v1.0.tsv

Reading TFLink_Mus_musculus_interactions_All_simpleFormat_v1.0.tsv...
First few lines of the file:
Line 1: UniprotID.TF	UniprotID.Target	NCBI.GeneID.TF	NCBI.GeneID.Target	Name.TF	Name.Target	Detection.method	PubmedID	Organism	Source.database	Small-scale.evidence	TF.TFLink.ortho	TF.nonTFLink.ortho	Target.TFLink.ortho	Target.nonTFLink.ortho
Line 2: Q9JL61	P14483	53970	14961	Rfx5	H2-Ab1	inferred by curator	29087512;11258423	Mus musculus	TRRUST	Yes	Hs:P48382;Rn:D3ZHD7	-	Rn:Q6AYB1	Hs:Q5SU54
Line 3: Q04207	Q38HM4	19697	-	Rela	Trim63	inferred by curator	24064732;29087512	Mus musculus	TRRUST	Yes	Hs:Q04206;Rn:Q7TQN4	-	-	-
Line 4: Q01147	P16014	12912	12653	Creb1	Chgb	inferred by curator	17202159	Mus musculus	TRED	Yes	Hs:P16220;Dr:Q803K8;Rn:P15337	Dr:E9QHG0	Hs:P05060;Rn:O35314	Dr:A5WWH0
Line 5: P51960	P48379	17864	19725	Mybl1	Rfx2	chromatin immunoprecipitation assay;inferred by curator	29087512

In [6]:
# ---------- Step 7: Extract ALL interactions with simplified columns ----------
if 'df_full' in locals() and not df_full.empty:
    print("Extracting all TF interactions with simplified columns...")
    
    # Check what columns are available
    print(f"Available columns: {list(df_full.columns)}")
    
    # Map the column names to what we need
    # The TFLink format might have different column names, let's find the right ones
    column_mapping = {}
    
    # Look for TF name columns
    for col in df_full.columns:
        if 'tf' in col.lower() and 'name' in col.lower():
            column_mapping['Name.TF'] = col
        elif 'target' in col.lower() and 'name' in col.lower():
            column_mapping['Name.Target'] = col
        elif 'detection' in col.lower() and 'method' in col.lower():
            column_mapping['Detection.method'] = col
        elif 'evidence' in col.lower() and 'type' in col.lower():
            column_mapping['Detection.method'] = col
    
    print(f"Column mapping found: {column_mapping}")
    
    # If we can't find the exact columns, let's try alternative names
    if not column_mapping:
        print("Trying alternative column names...")
        for col in df_full.columns:
            print(f"  Column: {col}")
        
        # Try to identify columns by content or position
        if len(df_full.columns) >= 3:
            # Often the first few columns contain the main information
            potential_tf_col = df_full.columns[0] if 'tf' in str(df_full.columns[0]).lower() else None
            potential_target_col = df_full.columns[1] if 'target' in str(df_full.columns[1]).lower() else None
            potential_method_col = None
            
            # Look for method/evidence columns
            for col in df_full.columns:
                if any(keyword in str(col).lower() for keyword in ['method', 'evidence', 'assay', 'type']):
                    potential_method_col = col
                    break
            
            column_mapping = {
                'Name.TF': potential_tf_col or df_full.columns[0],
                'Name.Target': potential_target_col or df_full.columns[1],
                'Detection.method': potential_method_col or 'chromatin immunoprecipitation assay'
            }
    
    print(f"Using column mapping: {column_mapping}")
    
    # Extract the data
    if all(col in df_full.columns for col in column_mapping.values()):
        # Extract only the columns we need
        df_simplified_all = df_full[list(column_mapping.values())].copy()
        
        # Rename columns to our standard format
        df_simplified_all.columns = ['Name.TF', 'Name.Target', 'Detection.method']
        
        # Remove any rows with missing data
        df_simplified_all = df_simplified_all.dropna()
        
        # Remove duplicates
        df_simplified_all = df_simplified_all.drop_duplicates()
        
        print(f"Extracted {len(df_simplified_all)} unique TF interactions")
        print(f"Shape: {df_simplified_all.shape}")
        print(f"\nFirst 10 interactions:")
        print(df_simplified_all.head(10))
        
        # Save to CSV
        output_file_all = os.path.join(tf_dir, "all_tf_interactions_simplified.csv")
        df_simplified_all.to_csv(output_file_all, index=False)
        
        print(f"\nAll TF interactions saved to: {output_file_all}")
        print(f"Total interactions: {len(df_simplified_all)}")
        
    else:
        print("Could not find the required columns. Let's examine the data structure more:")
        print(df_full.head())
        print(f"\nColumn names: {list(df_full.columns)}")
        
else:
    print("No data loaded. Please run the previous cell first.")

Extracting all TF interactions with simplified columns...
Available columns: ['UniprotID.TF', 'UniprotID.Target', 'NCBI.GeneID.TF', 'NCBI.GeneID.Target', 'Name.TF', 'Name.Target', 'Detection.method', 'PubmedID', 'Organism', 'Source.database', 'Small-scale.evidence', 'TF.TFLink.ortho', 'TF.nonTFLink.ortho', 'Target.TFLink.ortho', 'Target.nonTFLink.ortho']
Column mapping found: {'Name.TF': 'Name.TF', 'Name.Target': 'Name.Target', 'Detection.method': 'Detection.method'}
Using column mapping: {'Name.TF': 'Name.TF', 'Name.Target': 'Name.Target', 'Detection.method': 'Detection.method'}
Extracted 4023181 unique TF interactions
Shape: (4023181, 3)

First 10 interactions:
  Name.TF Name.Target                                   Detection.method
0    Rfx5      H2-Ab1                                inferred by curator
1    Rela      Trim63                                inferred by curator
2   Creb1        Chgb                                inferred by curator
3   Mybl1        Rfx2  chromatin imm

In [7]:
import pandas as pd

df = df_simplified_all.copy()

# Normalize text (safe)
df["Detection.method"] = df["Detection.method"].astype(str).str.lower().str.strip()

# Split methods by semicolon and check if ALL methods are "inferred by curator"
def has_only_inferred(method_string):
    """Check if all methods in the string are 'inferred by curator'"""
    methods = [m.strip() for m in method_string.split(';')]
    # Return True if ALL methods are "inferred by curator"
    return all(m == "inferred by curator" for m in methods)

# Flag interactions that have ONLY "inferred by curator" (no experimental evidence)
df["only_inferred"] = df["Detection.method"].apply(has_only_inferred)

# Keep only interactions that have at least one experimental method
df_filtered = df[~df["only_inferred"]].copy()

# Drop helper column
df_filtered = df_filtered.drop(columns=["only_inferred"])

print(f"Original interactions: {len(df)}")
print(f"Filtered interactions: {len(df_filtered)}")
print(f"Removed interactions: {len(df) - len(df_filtered)}")

# Show some examples of what was kept
print("\nExample of kept interactions (with experimental evidence):")
print(df_filtered.head(10))

# Show some examples of what was removed
print("\nExample of removed interactions (only inferred):")
removed = df[df["only_inferred"]]
print(removed.head(10))

# Save
df_filtered.to_csv("./tf_link_data/all_tf_interactions_experimental_only.csv", index=False)
print("\nSaved filtered interactions.")

Original interactions: 4023181
Filtered interactions: 4005485
Removed interactions: 17696

Example of kept interactions (with experimental evidence):
   Name.TF Name.Target                                   Detection.method
3    Mybl1        Rfx2  chromatin immunoprecipitation assay;inferred b...
4    Hif1a         Bax  chromatin immunoprecipitation assay;inferred b...
5      Jun    B4galnt1  chromatin immunoprecipitation assay;inferred b...
6    Ppara        Pdk4  chromatin immunoprecipitation assay;inferred b...
9   Nfe2l2        Cd36  chromatin immunoprecipitation assay;inferred b...
12   Pparg         Dbi  chromatin immunoprecipitation assay;inferred b...
16     Jun        Il11  chromatin immunoprecipitation assay;inferred b...
21     Yy1         Egf  chromatin immunoprecipitation assay;inferred b...
23     Sp1        Tal1  chromatin immunoprecipitation assay;inferred b...
24   Hif1a         Bsg  chromatin immunoprecipitation assay;inferred b...

Example of removed interactions (on